# L14 – Samma sak i TensorFlow/Keras

**Tid:** ca 1,5 h · **Del D: Avslutning**

## Mål
1. Känna igen **samma begrepp** i Keras som i PyTorch: lager, loss, optimerare, epoker, early stopping och schedulers.
2. Bygga och träna MLP:n från L05 med `compile` och `fit`.
3. Förstå skillnaden mellan Keras färdiga `fit` och den egna loopen i PyTorch, och **när** man väljer vilket.

**Var i arbetsflödet?** Arbetsflödet är detsamma. Bara verktyget byts ut.

> TensorFlow och Keras finns redan i repots `.venv` (från AI25-kursen). TensorFlow har inte stöd för GTX 1080 i alla byggen. Om ingen GPU hittas körs koden på CPU, och för den här lilla modellen tar det bara någon minut. **Körtid:** cirka 2–4 minuter.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"   # Keras 3 kan köra på flera backends. Vi väljer TensorFlow.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"     # mindre loggbrus från TensorFlow

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers

keras.utils.set_random_seed(42)              # motsvarar set_seed: Python, NumPy och TensorFlow
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU:er som TensorFlow ser:", tf.config.list_physical_devices("GPU"))

I Keras sköts placeringen på `device` automatiskt. Det finns ingen `.to(device)`. TensorFlow använder GPU:n om den hittas och annars CPU:n.

## 1. Data: Fashion-MNIST igen

Keras har en egen nedladdning av Fashion-MNIST. Den ger NumPy-arrayer i stället för en `Dataset`. Vi följer samma arbetsflöde: dela upp och normalisera med statistik från train.

In [ ]:
(x_full, y_full), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
print("x_full:", x_full.shape, x_full.dtype)          # (60000, 28, 28) uint8

rng = np.random.default_rng(42)
perm = rng.permutation(len(x_full))
train_idx, val_idx = perm[:50_000], perm[50_000:]
x_train, y_train = x_full[train_idx], y_full[train_idx]
x_val, y_val = x_full[val_idx], y_full[val_idx]

MEAN, STD = (x_train / 255.0).mean(), (x_train / 255.0).std()   # bara train
normalize = lambda x: ((x / 255.0 - MEAN) / STD).astype("float32")
x_train, x_val, x_test = normalize(x_train), normalize(x_val), normalize(x_test)
print(f"MEAN={MEAN:.4f} STD={STD:.4f}  train {x_train.shape}  val {x_val.shape}  test {x_test.shape}")

## 2. Bygg modellen

Samma arkitektur som i L05: 784, 256, 128 och 10. Jämför sida vid sida:

| PyTorch | Keras |
|---|---|
| `nn.Flatten()` | `layers.Flatten()` |
| `nn.Linear(784, 256)` och `nn.ReLU()` | `layers.Dense(256, activation="relu")`. Indatastorleken räknas ut automatiskt. |
| `nn.Linear(128, 10)` (logits) | `layers.Dense(10)` (ingen aktivering: logits) |

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(10),                       # logits, precis som i PyTorch
])
model.summary()

Antalet parametrar, **235 146**, är exakt detsamma som för PyTorch-modellen i L05. Det är samma modell.

## 3. `compile`: välj loss, optimerare och mått

I PyTorch skapade vi `loss_fn` och `optimizer` som egna objekt. I Keras kopplas de till modellen med `compile`:

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

- **`SparseCategoricalCrossentropy`** motsvarar `nn.CrossEntropyLoss`. *Sparse* betyder att etiketterna är klassindex (0–9), inte one-hot. Utan "Sparse" förväntar sig Keras one-hot.
- **`from_logits=True`** är viktigt. Det säger att modellen ger logits (L03). Standardvärdet i Keras är `False`, alltså sannolikheter. Glömmer du det med en modell som ger logits tränar den sämre utan felmeddelande, samma sorts tysta bugg som dubbel softmax i L09.

## 4. `fit` med callbacks

`model.fit` är hela träningsloopen från L03–L09 i ett anrop. Det som vi byggde in i `dlkit.fit` heter **callbacks** i Keras:

| `dlkit.fit` (PyTorch) | Keras callback |
|---|---|
| `patience=5` (early stopping och återställning av de bästa vikterna) | `EarlyStopping(patience=5, restore_best_weights=True)` |
| `scheduler=ReduceLROnPlateau(...)` | `ReduceLROnPlateau(factor=0.5, patience=2)` |
| `checkpoint_path=...` | `ModelCheckpoint(path, save_best_only=True)` |
| `writer=SummaryWriter(...)` | `TensorBoard(log_dir=...)` |

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
    keras.callbacks.TensorBoard(log_dir="../runs/keras_mlp"),
]
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),   # val används för early stopping och schedulern, som i dlkit
    epochs=30,
    batch_size=128,                   # Keras blandar träningsdatan varje epok som standard
    callbacks=callbacks,
    verbose=2,
)

`history.history` är en ordbok med samma struktur som vår `history` i `dlkit`:

In [ ]:
h = history.history
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric in zip(axes, ["loss", "accuracy"]):
    ax.plot(h[metric], "o-", label="train"); ax.plot(h[f"val_{metric}"], "o-", label="val")
    ax.set_xlabel("epok"); ax.set_title(metric); ax.legend(); ax.grid(alpha=0.3)
plt.show()
print(f"bästa val accuracy: {max(h['val_accuracy']):.3f}")

## 5. Utvärdera, prediktera och spara

In [ ]:
logits = model.predict(x_val, verbose=0)          # (10000, 10) logits
y_pred = logits.argmax(axis=1)

from sklearn.metrics import classification_report
print(classification_report(y_val, y_pred, digits=3))   # samma sklearn-utvärdering som i L06

In [ ]:
os.makedirs("../checkpoints", exist_ok=True)
model.save("../checkpoints/keras_mlp.keras")           # arkitektur och vikter i ett format
loaded = keras.models.load_model("../checkpoints/keras_mlp.keras")
print("identiska prediktioner:", np.array_equal(loaded.predict(x_val, verbose=0).argmax(1), y_pred))

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)   # test en gång
print(f"TEST accuracy: {test_acc:.3f}")

I Keras sparar man ofta **hela modellen**, arkitektur och vikter, i `.keras`-formatet. Det är ett säkert format byggt för ändamålet, inte pickle (jämför L09).

## 6. PyTorch kontra Keras – när väljer jag vad?

| | PyTorch | Keras |
|---|---|---|
| Träningsloop | du skriver den själv: full kontroll och lätt att felsöka | `fit()`: kortare kod och mindre att göra fel |
| Standardfall (klassificering, regression) | mer kod | **snabbt och enkelt** |
| Ovanliga upplägg (egna loss, flera modeller, forskning) | **flexibelt** | går, men kräver mer arbete (`train_step`) |
| Ekosystem | dominerar forskning och Hugging Face | stark i produktion (TF Serving, TF Lite för mobil) |
| Bildformat | kanalerna först: `(N, C, H, W)` | **kanalerna sist**: `(N, H, W, C)` |

**Viktigast:** begreppen är desamma. Om du förstår loss, optimerare, learning rate, batchar, epoker, val, early stopping och regularisering kan du byta ramverk på en eftermiddag. Det var poängen med att bygga loopen själv i PyTorch först.

## 7. Övningar

### Övning 1 – regularisering i Keras
Lägg till `layers.Dropout(...)` och `layers.BatchNormalization()` i modellen (L07) och träna med samma callbacks. Nå minst **89 % val accuracy**.

*Tänk på:* Keras växlar automatiskt mellan train- och eval-läge. `fit` använder train-läge och `evaluate` och `predict` eval-läge.

In [ ]:
keras.utils.set_random_seed(42)
# TODO
reg_history = None

In [ ]:
assert max(reg_history.history["val_accuracy"]) >= 0.89, max(reg_history.history["val_accuracy"])
print("Rätt!", round(max(reg_history.history["val_accuracy"]), 3))

### Övning 2 – en CNN i Keras
Bygg en liten CNN, med `layers.Conv2D` och `layers.MaxPooling2D` som i L10, för Fashion-MNIST, och nå minst **90 % val accuracy**.

*Ledtråd:* Keras vill ha **kanalerna sist**, så bilderna måste ha shape `(N, 28, 28, 1)`. Använd `x_train[..., None]`.

In [ ]:
keras.utils.set_random_seed(42)
# TODO
cnn_history = None

In [ ]:
assert max(cnn_history.history["val_accuracy"]) >= 0.90, max(cnn_history.history["val_accuracy"])
print("Rätt!", round(max(cnn_history.history["val_accuracy"]), 3))

### Övning 3 – reflektion
1. Vilken del av `dlkit.fit` motsvarar `validation_data` och `EarlyStopping` i Keras?
2. Vad händer om du glömmer `from_logits=True`?
3. Du ska bygga en vanlig bildklassificerare för en app. Vilket ramverk väljer du, och varför? Ändras svaret om du ska implementera en ny forskningsidé?

*Dina svar:*

## När använder jag detta?
- **Keras `fit`:** standardproblem där du vill ha lite kod och färdiga callbacks, och projekt som redan använder TensorFlow.
- **PyTorch med egen loop:** när du behöver kontroll, felsöker eller gör något ovanligt, och när du vill använda forskningskod eller Hugging Face.
- **Båda:** samma arbetsflöde, samma begrepp och samma diagnos.

## Sammanfattning
- `Dense` motsvarar `Linear`, `compile` motsvarar att välja loss och optimerare, och `fit` motsvarar träningsloopen.
- Callbacks motsvarar `patience`, `scheduler`, `checkpoint_path` och `writer` i `dlkit`.
- Använd `from_logits=True` när modellen ger logits.
- Keras använder kanalerna sist och PyTorch kanalerna först.

**Nästa lektion:** miniprojektet, där du gör hela arbetsflödet själv.